# Tema 1.2 - CompraFácil
## Práctica guiada: de métricas técnicas a impacto de negocio

Este notebook está **100% homologado** con el estilo del cuaderno anterior y organizado por **pasos** de la práctica.

### Propósito de la práctica
En esta práctica aprenderás a traducir objetivos estratégicos de negocio en métricas analíticas que permitan evaluar el impacto real de un modelo predictivo.

Continuarás trabajando con el proyecto **CompraFácil**, donde en la sesión anterior se desarrolló un modelo para detectar el abandono de carritos de compra. Aunque dicho modelo obtuvo resultados técnicos favorables, antes de implementarlo en producción el equipo de marketing necesita determinar si la solución generará beneficios reales para la organización.

### Archivos de salida que generará este notebook
- `impact_metrics.csv`
- `threshold_sweep.csv`
- `kpi_breakdown.csv`


In [ ]:
# ============================================================
# 0. Configuración inicial
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
rng = np.random.default_rng(SEED)

display(Markdown("### Librerías cargadas correctamente"))
print("Entorno listo para la práctica 2 de CompraFácil.")


## Paso 1. Cargar las métricas del modelo

Comienza importando los archivos generados en el cuaderno anterior: `metrics.csv` y `predictions.csv`.

Si no cuentas con ellos en tu entorno de Colab, este notebook incluye una **simulación de respaldo** para que puedas continuar sin contratiempos.

Al ejecutar la siguiente sección podrás revisar:
- precisión,
- recall,
- F1 Score,
- AUC,
- y las predicciones que servirán de base para conectar desempeño técnico con impacto económico.


In [ ]:
# ============================================================
# 1. Carga de metrics.csv y predictions.csv
#    Si no existen, se simulan para poder continuar.
# ============================================================

metrics_path = Path("metrics.csv")
predictions_path = Path("predictions.csv")

if metrics_path.exists() and predictions_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    predictions_df = pd.read_csv(predictions_path)
    source_mode = "archivos reales"
else:
    # --------------------------------------------------------
    # Simulación de respaldo
    # --------------------------------------------------------
    n = 1800
    y_true = rng.binomial(1, 0.42, size=n)

    # Distribuciones con separación razonable entre clases
    y_proba = np.where(
        y_true == 1,
        rng.beta(5, 2, size=n),   # clientes con mayor propensión a abandonar
        rng.beta(2, 5, size=n)    # clientes con menor propensión
    )

    predictions_df = pd.DataFrame({
        "customer_id": np.arange(1, n + 1),
        "y_true": y_true,
        "y_proba": y_proba
    })

    predictions_df["y_pred_default"] = (predictions_df["y_proba"] >= 0.50).astype(int)
    predictions_df["model"] = "logistic_regression"

    metrics_df = pd.DataFrame([{
        "model": "logistic_regression",
        "precision": round(float(precision_score(predictions_df["y_true"], predictions_df["y_pred_default"])), 4),
        "recall": round(float(recall_score(predictions_df["y_true"], predictions_df["y_pred_default"])), 4),
        "f1_score": round(float(f1_score(predictions_df["y_true"], predictions_df["y_pred_default"])), 4),
        "auc": round(float(roc_auc_score(predictions_df["y_true"], predictions_df["y_proba"])), 4),
        "train_size": 3750,
        "test_size": len(predictions_df),
        "default_threshold": 0.50,
        "selected_model": True
    }])

    source_mode = "simulación de respaldo"

display(Markdown(f"### Fuente de datos utilizada: **{source_mode}**"))

display(Markdown("### Métricas del modelo"))
display(metrics_df)

display(Markdown("### Predicciones disponibles"))
display(predictions_df.head())

print("Dimensiones de predictions_df:", predictions_df.shape)


## Paso 2. Definir los parámetros del negocio

Ahora establecerás algunos supuestos económicos para cuantificar el impacto financiero del modelo.

Estos parámetros representan, por ejemplo:
- cuánto gana la empresa al recuperar un carrito,
- cuánto cuesta intervenir a un cliente,
- y qué proporción de clientes realmente responde a la intervención.

Recuerda: estos factores pueden afectar el ROI incluso más que una mejora marginal en las métricas técnicas.


In [ ]:
# ============================================================
# 2. Parámetros del negocio
# ============================================================

business_params = {
    "revenue_per_recovered_cart": 850.0,   # ingreso promedio por carrito recuperado
    "intervention_cost": 35.0,             # costo por enviar recordatorio/cupón/notificación
    "adoption_rate": 0.18,                 # proporción de clientes rescatables que reaccionan a la campaña
    "max_campaign_volume": 0.45,           # tope de clientes a intervenir por campaña
    "precision_floor": 0.50,               # precisión mínima aceptable
    "recall_floor": 0.55                   # recall mínimo aceptable
}

business_df = pd.DataFrame({
    "Parámetro": list(business_params.keys()),
    "Valor": list(business_params.values())
})

display(Markdown("### Parámetros de negocio definidos"))
display(business_df)

print("Supuestos económicos listos.")


## Paso 3. Calcular indicadores de negocio

En esta fase evaluarás diferentes **umbrales de decisión** para identificar el punto que maximiza el **ROI** sin comprometer de manera crítica la calidad predictiva.

La lógica es la siguiente:
1. recorrer varios thresholds,
2. calcular métricas técnicas,
3. estimar carritos recuperados, beneficio bruto, costo de intervención y beneficio neto,
4. identificar el umbral óptimo desde la perspectiva de negocio.

En la práctica profesional, este análisis es crucial, porque un modelo ligeramente menos preciso puede generar un ROI significativamente mayor.


In [ ]:
# ============================================================
# 3. Evaluación por thresholds y cálculo de ROI
# ============================================================

def evaluate_thresholds(preds: pd.DataFrame, params: dict, thresholds=None) -> pd.DataFrame:
    if thresholds is None:
        thresholds = np.arange(0.10, 0.91, 0.05)

    rows = []
    total_population = len(preds)

    for threshold in thresholds:
        y_pred = (preds["y_proba"] >= threshold).astype(int)
        y_true = preds["y_true"]

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        auc = roc_auc_score(y_true, preds["y_proba"])

        targeted = int(y_pred.sum())
        campaign_share = targeted / total_population if total_population > 0 else 0

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())

        recovered_carts = tp * params["adoption_rate"]
        gross_benefit = recovered_carts * params["revenue_per_recovered_cart"]
        intervention_cost_total = targeted * params["intervention_cost"]
        net_benefit = gross_benefit - intervention_cost_total

        if intervention_cost_total > 0:
            roi = (net_benefit / intervention_cost_total) * 100
        else:
            roi = 0.0

        feasible = (
            (precision >= params["precision_floor"]) and
            (recall >= params["recall_floor"]) and
            (campaign_share <= params["max_campaign_volume"])
        )

        rows.append({
            "threshold": round(float(threshold), 2),
            "precision": round(float(precision), 4),
            "recall": round(float(recall), 4),
            "f1_score": round(float(f1), 4),
            "auc": round(float(auc), 4),
            "targeted_customers": targeted,
            "campaign_share": round(float(campaign_share), 4),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "recovered_carts_est": round(float(recovered_carts), 2),
            "gross_benefit": round(float(gross_benefit), 2),
            "intervention_cost_total": round(float(intervention_cost_total), 2),
            "net_benefit": round(float(net_benefit), 2),
            "roi_pct": round(float(roi), 2),
            "feasible": feasible
        })

    return pd.DataFrame(rows)

threshold_sweep_df = evaluate_thresholds(predictions_df, business_params)

display(Markdown("### Sensibilidad del modelo ante distintos thresholds"))
display(threshold_sweep_df.head(10))

feasible_df = threshold_sweep_df[threshold_sweep_df["feasible"]].copy()

if len(feasible_df) > 0:
    best_row = feasible_df.sort_values(
        by=["roi_pct", "f1_score", "precision"],
        ascending=False
    ).iloc[0]
else:
    best_row = threshold_sweep_df.sort_values(
        by=["roi_pct", "f1_score", "precision"],
        ascending=False
    ).iloc[0]

best_threshold = float(best_row["threshold"])

display(Markdown(f"### Umbral óptimo identificado: **{best_threshold:.2f}**"))
display(pd.DataFrame([best_row]))

print("Threshold óptimo calculado correctamente.")


## Paso 4. Visualizar los resultados

Ahora generarás visualizaciones para interpretar la relación entre:
- desempeño técnico,
- volumen de clientes a intervenir,
- y rentabilidad del modelo.

Analiza especialmente si existe un equilibrio razonable entre **F1 Score**, **Recall** y **ROI**.


In [ ]:
# ============================================================
# 4. Visualizaciones comparativas
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["roi_pct"], marker="o")
plt.axvline(best_threshold, linestyle="--")
plt.title("ROI por threshold")
plt.xlabel("Threshold")
plt.ylabel("ROI (%)")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], marker="o", label="Precision")
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], marker="o", label="Recall")
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1_score"], marker="o", label="F1 Score")
plt.axvline(best_threshold, linestyle="--", label="Threshold óptimo")
plt.title("Métricas técnicas por threshold")
plt.xlabel("Threshold")
plt.ylabel("Valor")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["campaign_share"], marker="o")
plt.axvline(best_threshold, linestyle="--")
plt.title("Proporción de clientes intervenidos por threshold")
plt.xlabel("Threshold")
plt.ylabel("Campaign share")
plt.grid(True, alpha=0.3)
plt.show()


## Paso 5. Exportar los resultados a Power BI

Para elaborar un tablero ejecutivo, exportarás los principales resultados del análisis.  
El notebook generará tres archivos:

- `impact_metrics.csv` → modelo recomendado, umbral óptimo y ROI.
- `threshold_sweep.csv` → sensibilidad del beneficio ante distintos umbrales.
- `kpi_breakdown.csv` → resumen de métricas para construcción del dashboard.


In [ ]:
# ============================================================
# 5. Construcción de archivos de salida
# ============================================================

selected_model = (
    metrics_df.loc[metrics_df["selected_model"] == True, "model"].iloc[0]
    if "selected_model" in metrics_df.columns and metrics_df["selected_model"].eq(True).any()
    else metrics_df["model"].iloc[0]
)

impact_metrics_df = pd.DataFrame([{
    "selected_model": selected_model,
    "optimal_threshold": round(best_threshold, 2),
    "precision_optimal": float(best_row["precision"]),
    "recall_optimal": float(best_row["recall"]),
    "f1_score_optimal": float(best_row["f1_score"]),
    "auc_model": float(best_row["auc"]),
    "targeted_customers": int(best_row["targeted_customers"]),
    "campaign_share": float(best_row["campaign_share"]),
    "estimated_recovered_carts": float(best_row["recovered_carts_est"]),
    "gross_benefit": float(best_row["gross_benefit"]),
    "intervention_cost_total": float(best_row["intervention_cost_total"]),
    "net_benefit": float(best_row["net_benefit"]),
    "roi_pct": float(best_row["roi_pct"]),
    "precision_floor": business_params["precision_floor"],
    "recall_floor": business_params["recall_floor"],
    "max_campaign_volume": business_params["max_campaign_volume"]
}])

kpi_breakdown_df = pd.DataFrame([
    {"kpi": "precision_optimal", "value": float(best_row["precision"])},
    {"kpi": "recall_optimal", "value": float(best_row["recall"])},
    {"kpi": "f1_score_optimal", "value": float(best_row["f1_score"])},
    {"kpi": "auc_model", "value": float(best_row["auc"])},
    {"kpi": "targeted_customers", "value": int(best_row["targeted_customers"])},
    {"kpi": "campaign_share", "value": float(best_row["campaign_share"])},
    {"kpi": "estimated_recovered_carts", "value": float(best_row["recovered_carts_est"])},
    {"kpi": "gross_benefit", "value": float(best_row["gross_benefit"])},
    {"kpi": "intervention_cost_total", "value": float(best_row["intervention_cost_total"])},
    {"kpi": "net_benefit", "value": float(best_row["net_benefit"])},
    {"kpi": "roi_pct", "value": float(best_row["roi_pct"])}
])

impact_metrics_df.to_csv("impact_metrics.csv", index=False)
threshold_sweep_df.to_csv("threshold_sweep.csv", index=False)
kpi_breakdown_df.to_csv("kpi_breakdown.csv", index=False)

display(Markdown("### Archivo: impact_metrics.csv"))
display(impact_metrics_df)

display(Markdown("### Archivo: threshold_sweep.csv"))
display(threshold_sweep_df.head())

display(Markdown("### Archivo: kpi_breakdown.csv"))
display(kpi_breakdown_df)

print("Archivos exportados correctamente:")
print("- impact_metrics.csv")
print("- threshold_sweep.csv")
print("- kpi_breakdown.csv")

try:
    from google.colab import files
    files.download("impact_metrics.csv")
    files.download("threshold_sweep.csv")
    files.download("kpi_breakdown.csv")
except Exception:
    print("Si no estás en Google Colab, ignora este mensaje. Los archivos ya fueron creados en el directorio de trabajo.")


## Paso 6. Validación y análisis

Con base en los resultados obtenidos, reflexiona sobre estas preguntas:

1. ¿Qué implicaciones tendría aplicar el umbral recomendado en una campaña real?
2. ¿El ROI obtenido justificaría el costo de implementación del modelo?
3. ¿Qué ajustes podrías realizar para mejorar la tasa de adopción o reducir costos operativos?

En la práctica profesional, esta fase es fundamental porque conecta las métricas del modelo con la toma de decisiones estratégicas.


In [ ]:
# ============================================================
# 6. Resumen ejecutivo automático
# ============================================================

summary_lines = [
    f"Modelo seleccionado: {selected_model}",
    f"Threshold óptimo: {best_threshold:.2f}",
    f"Precision en el threshold óptimo: {best_row['precision']:.4f}",
    f"Recall en el threshold óptimo: {best_row['recall']:.4f}",
    f"F1 Score en el threshold óptimo: {best_row['f1_score']:.4f}",
    f"ROI estimado: {best_row['roi_pct']:.2f}%",
    f"Clientes intervenidos: {int(best_row['targeted_customers'])}",
    f"Beneficio neto estimado: ${best_row['net_benefit']:,.2f}"
]

display(Markdown("### Resumen ejecutivo"))
for line in summary_lines:
    print("-", line)

display(Markdown(
    '''
### Conclusión
Este notebook permitió traducir el desempeño técnico del modelo en un análisis de valor de negocio.  
A partir de distintos thresholds se identificó el punto de decisión que maximiza el ROI bajo restricciones operativas y de calidad.
'''
))


## Resumen final de la práctica

Este cuaderno:
1. carga `metrics.csv` y `predictions.csv` o genera una simulación de respaldo,
2. define parámetros económicos del caso CompraFácil,
3. evalúa distintos umbrales de decisión,
4. identifica el threshold óptimo con base en ROI y restricciones,
5. visualiza la relación entre métricas técnicas y métricas financieras,
6. exporta tres archivos para Power BI o tablero ejecutivo:
   - `impact_metrics.csv`
   - `threshold_sweep.csv`
   - `kpi_breakdown.csv`
